In [1]:
# 🎯 Scenario: Medical Image Classification
# You’re training a convolutional neural network (CNN) to detect pneumonia from chest X-rays.
# - Training accuracy: 95%
# - Validation accuracy: 74%
# At first glance, the model seems powerful — it almost perfectly classifies the training set. But the sharp drop in validation accuracy signals overfitting: the network has memorized the training images (specific pixel patterns, noise, or even hospital-specific artifacts) instead of learning generalizable features of pneumonia.

# ⚙️ Levers to Address Overfitting
# - Data Augmentation: Rotate, flip, and adjust brightness of X-rays to simulate variability.
# - Regularization: Apply dropout in dense layers or L2 weight decay.
# - Transfer Learning: Use a pretrained backbone (e.g., ResNet) to leverage generalized features.
# - Cross-validation: Ensure robustness across different patient subsets.
# - Early Stopping: Halt training when validation loss stops improving.

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bl9cksheep/medical-image")

print("Path to dataset files:", path)

100%|██████████| 19.8M/19.8M [00:00<00:00, 104MB/s] 


Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/bl9cksheep/medical-image/versions/3


In [5]:
# Medical Image Classification (Pneumonia Detection) with Overfitting Control

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import numpy as np

# ---------------------------
# Device (GPU if available)
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------
# 1. Data Augmentation
# ---------------------------
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# ---------------------------
# 2. Load Dataset
# ---------------------------
dataset_path = "/root/.cache/kagglehub/datasets/bl9cksheep/medical-image/versions/3"

dataset = datasets.ImageFolder(dataset_path, transform=train_transform)

# Number of classes
num_classes = len(dataset.classes)
print("Classes:", dataset.classes)

# ---------------------------
# 3. Train / Validation Split
# ---------------------------
indices = list(range(len(dataset)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

train_subset = Subset(dataset, train_idx)
val_subset = Subset(dataset, val_idx)

# Different transforms for train and validation
train_subset.dataset.transform = train_transform
val_subset.dataset.transform = val_transform

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=32)

# ---------------------------
# 4. Transfer Learning (ResNet18)
# ---------------------------
model = resnet18(weights=ResNet18_Weights.DEFAULT)

num_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_features, num_classes)
)

model = model.to(device)

# ---------------------------
# 5. Loss and Optimizer
# ---------------------------
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4   # L2 Regularization
)

# ---------------------------
# 6. Training with Early Stopping
# ---------------------------
best_val_loss = float("inf")
patience = 5
counter = 0
epochs = 20

for epoch in range(epochs):

    # -------- Training --------
    model.train()
    train_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    # -------- Validation --------
    model.eval()
    val_loss = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # -------- Early Stopping --------
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        counter = 0

    else:

        counter += 1

        if counter >= patience:
            print("Early stopping triggered")
            break

Classes: ['data', 'enhanced_data', 'test']
Epoch 1 | Train Loss: 2.5684 | Val Loss: 9.0448
Epoch 2 | Train Loss: 0.3076 | Val Loss: 32.4881
Epoch 3 | Train Loss: 0.2046 | Val Loss: 9.1149
Epoch 4 | Train Loss: 0.1518 | Val Loss: 2.1594
Epoch 5 | Train Loss: 0.0201 | Val Loss: 0.2500
Epoch 6 | Train Loss: 0.0406 | Val Loss: 0.2553
Epoch 7 | Train Loss: 0.0095 | Val Loss: 0.2511
Epoch 8 | Train Loss: 0.0120 | Val Loss: 0.2437
Epoch 9 | Train Loss: 0.0037 | Val Loss: 0.2479
Epoch 10 | Train Loss: 0.0030 | Val Loss: 0.2494
Epoch 11 | Train Loss: 0.0040 | Val Loss: 0.2568
Epoch 12 | Train Loss: 0.0006 | Val Loss: 0.2690
Epoch 13 | Train Loss: 0.0011 | Val Loss: 0.2839
Early stopping triggered
